### Feature Engineering 

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import holidays

In [2]:
df = pd.read_csv("/Users/saif/Desktop/ML_Load_Forecasting/data/Silver/df_clean.csv", index_col=0, parse_dates= True)

In [3]:
df.head()

,Load_MW,temperature_2m,wind_speed_10m,shortwave_radiation
2020-01-01 00:00:00,43471.87,1.5,11.4,0.0
2020-01-01 01:00:00,42401.86,2.0,10.3,0.0
2020-01-01 02:00:00,41126.54,1.8,11.2,0.0
2020-01-01 03:00:00,40793.28,1.1,10.1,0.0
2020-01-01 04:00:00,40710.44,0.3,9.4,0.0


### Time features

In [4]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["is_weekend"] = (df.dayofweek >= 5).astype(int)


### adding fourier features

In [5]:
df["hour_sin"] = np.sin(2 * np.pi * df.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df.hour / 24)
df["day_sin"] = np.sin(2 * np.pi * df.dayofweek / 7)
df["day_cos"] = np.cos(2 * np.pi * df.dayofweek / 7)
df["month_sin"] = np.sin(2 * np.pi * df.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df.month / 12)


In [6]:
de_holidays = holidays.Germany(years = (2020, 2024))

df["is_holiday"] = df.index.normalize().isin(de_holidays).astype(int)

print(df.head())
print(df.describe())

                      Load_MW  temperature_2m  wind_speed_10m  \
2020-01-01 00:00:00  43471.87             1.5            11.4   
2020-01-01 01:00:00  42401.86             2.0            10.3   
2020-01-01 02:00:00  41126.54             1.8            11.2   
2020-01-01 03:00:00  40793.28             1.1            10.1   
2020-01-01 04:00:00  40710.44             0.3             9.4   

                     shortwave_radiation  hour  dayofweek  month  is_weekend  \
2020-01-01 00:00:00                  0.0     0          2      1           0   
2020-01-01 01:00:00                  0.0     1          2      1           0   
2020-01-01 02:00:00                  0.0     2          2      1           0   
2020-01-01 03:00:00                  0.0     3          2      1           0   
2020-01-01 04:00:00                  0.0     4          2      1           0   

                     hour_sin  hour_cos   day_sin   day_cos  month_sin  \
2020-01-01 00:00:00  0.000000  1.000000  0.974928 -0.2

### Lag Features

In [7]:
df["lag_hour"] = df["Load_MW"].shift(-1)
df["lag_day"] = df["Load_MW"].shift(-24)
df["lag_week"] = df["Load_MW"].shift(-168)

In [8]:
print(df.head())
print(df.describe())

                      Load_MW  temperature_2m  wind_speed_10m  \
2020-01-01 00:00:00  43471.87             1.5            11.4   
2020-01-01 01:00:00  42401.86             2.0            10.3   
2020-01-01 02:00:00  41126.54             1.8            11.2   
2020-01-01 03:00:00  40793.28             1.1            10.1   
2020-01-01 04:00:00  40710.44             0.3             9.4   

                     shortwave_radiation  hour  dayofweek  month  is_weekend  \
2020-01-01 00:00:00                  0.0     0          2      1           0   
2020-01-01 01:00:00                  0.0     1          2      1           0   
2020-01-01 02:00:00                  0.0     2          2      1           0   
2020-01-01 03:00:00                  0.0     3          2      1           0   
2020-01-01 04:00:00                  0.0     4          2      1           0   

                     hour_sin  hour_cos   day_sin   day_cos  month_sin  \
2020-01-01 00:00:00  0.000000  1.000000  0.974928 -0.2

In [9]:
print(df.isna().sum())

Load_MW                  0
temperature_2m           0
wind_speed_10m           0
shortwave_radiation      0
hour                     0
dayofweek                0
month                    0
is_weekend               0
hour_sin                 0
hour_cos                 0
day_sin                  0
day_cos                  0
month_sin                0
month_cos                0
is_holiday               0
lag_hour                 1
lag_day                 24
lag_week               168
dtype: int64


In [10]:
df = df.dropna()
print(df.isna().sum())
print(df.shape)

Load_MW                0
temperature_2m         0
wind_speed_10m         0
shortwave_radiation    0
hour                   0
dayofweek              0
month                  0
is_weekend             0
hour_sin               0
hour_cos               0
day_sin                0
day_cos                0
month_sin              0
month_cos              0
is_holiday             0
lag_hour               0
lag_day                0
lag_week               0
dtype: int64
(34896, 18)


### Rolling Features

In [11]:
df["rolling_mean_24h"] = df["Load_MW"].rolling(window=24).mean()
df["rolling_mean_168h"] = df["Load_MW"].rolling(window=168).mean()
df["rolling_std_24h"] = df["Load_MW"].rolling(window=24).std()
df["rolling_std_168h"] = df["Load_MW"].rolling(window=168).std()
print(df.head())
print(df.describe())

                      Load_MW  temperature_2m  wind_speed_10m  \
2020-01-01 00:00:00  43471.87             1.5            11.4   
2020-01-01 01:00:00  42401.86             2.0            10.3   
2020-01-01 02:00:00  41126.54             1.8            11.2   
2020-01-01 03:00:00  40793.28             1.1            10.1   
2020-01-01 04:00:00  40710.44             0.3             9.4   

                     shortwave_radiation  hour  dayofweek  month  is_weekend  \
2020-01-01 00:00:00                  0.0     0          2      1           0   
2020-01-01 01:00:00                  0.0     1          2      1           0   
2020-01-01 02:00:00                  0.0     2          2      1           0   
2020-01-01 03:00:00                  0.0     3          2      1           0   
2020-01-01 04:00:00                  0.0     4          2      1           0   

                     hour_sin  hour_cos  ...  month_sin  month_cos  \
2020-01-01 00:00:00  0.000000  1.000000  ...        0.5   

In [12]:
print(df.isna().sum())
print(df.shape)


Load_MW                  0
temperature_2m           0
wind_speed_10m           0
shortwave_radiation      0
hour                     0
dayofweek                0
month                    0
is_weekend               0
hour_sin                 0
hour_cos                 0
day_sin                  0
day_cos                  0
month_sin                0
month_cos                0
is_holiday               0
lag_hour                 0
lag_day                  0
lag_week                 0
rolling_mean_24h        23
rolling_mean_168h      167
rolling_std_24h         23
rolling_std_168h       167
dtype: int64
(34896, 22)


In [13]:

df.dropna(inplace=True)
print(df.shape)
print(df.isnull().sum())

(34729, 22)
Load_MW                0
temperature_2m         0
wind_speed_10m         0
shortwave_radiation    0
hour                   0
dayofweek              0
month                  0
is_weekend             0
hour_sin               0
hour_cos               0
day_sin                0
day_cos                0
month_sin              0
month_cos              0
is_holiday             0
lag_hour               0
lag_day                0
lag_week               0
rolling_mean_24h       0
rolling_mean_168h      0
rolling_std_24h        0
rolling_std_168h       0
dtype: int64


In [14]:
df.columns = df.columns.str.strip()
df.to_csv("/Users/saif/Desktop/ML_Load_Forecasting/data/Gold/df_features_fourier_time_encoding.csv")